# Phase 3: Core Algorithm Development 🧠⚙️

Welcome to the **Core Algorithms** stage. In this notebook, we will develop and verify the four core engines of **AgriSense AI**:
1. **Forecasting Engine**: Generate baseline future inputs (Weeks 289-292) and run predictions.
2. **Best Market Recommendation Engine**: Compute prices across districts to identify spatial arbitrage.
3. **Price Crash Alert Engine**: Scan all markets for upcoming price collapses (>10% drop).
4. **Climate Simulator Engine**: Alter weather features dynamically to simulate climate stress impacts on prices.
5. **Chatbot NLP Engine**: Set up keyword matching to parse farmer queries and fetch insights.

In [ ]:
# Cell 1: Imports and Model Loading
import pandas as pd
import numpy as np
import joblib
import os

# Check files
model_path = "price_forecast_rf.joblib"
dataset_path = "cleaned_dataset.csv"

if not os.path.exists(model_path) or not os.path.exists(dataset_path):
    raise FileNotFoundError("Please run Phase 1 and Phase 2 first.")

model = joblib.load(model_path)
df = pd.read_csv(dataset_path)

features = [
    'Market_id', 'Month_Num', 'Week_Of_Month', 'time_index', 'month_sin', 'month_cos',
    'Rainfall', 'rain_lag_1', 'rain_lag_2', 'rain_lag_4', 'rain_lag_8', 
    'rain_cum_4w', 'rain_cum_8w', 'rain_cum_12w', 'is_monsoon', 'is_dry', 
    'monsoon_total', 'monsoon_prev_year', 'temp_mean', 'temp_min', 'temp_max', 'temp_range', 
    'temp_mean_lag_1', 'temp_mean_lag_2', 'temp_mean_lag_3', 'temp_mean_lag_4', 
    'temp_range_lag_1', 'temp_range_lag_2', 'temp_mean_roll_2', 'temp_mean_roll_4', 
    'has_neighbors', 
    'neigh_price_lag_1', 'neigh_price_lag_2', 'neigh_price_lag_4', 
    'neigh_price_diff_lag_1', 'neigh_price_diff_lag_2', 'neigh_price_diff_lag_4'
]

print("Model loaded successfully!")
print(f"Data loaded. Unique markets: {df['Market'].nunique()}")

In [ ]:
# Cell 2: Forecasting Engine
def generate_forecast(market_name):
    """Generates 4 weeks of future baseline forecast (weeks 289 to 292) for a selected market"""
    market_df = df[df['Market'] == market_name]
    if market_df.empty:
        return pd.DataFrame()
    
    market_id = market_df['Market_id'].iloc[0]
    future_weeks = [289, 290, 291, 292]
    future_rows = []
    
    for t in future_weeks:
        year = 2020 + (t - 1) // 48
        week_in_year = (t - 1) % 48
        month = week_in_year // 4 + 1
        week = week_in_year % 4 + 1
        
        # Use average historical variables for the same calendar month and week
        hist_week = market_df[(market_df['Month_Num'] == month) & (market_df['Week_Of_Month'] == week)]
        if hist_week.empty:
            hist_week = market_df[market_df['Month_Num'] == month]
            
        mean_features = hist_week[features].mean()
        mean_features['Year'] = year
        mean_features['time_index'] = t
        mean_features['Market_id'] = market_id
        mean_features['Month_Num'] = month
        mean_features['Week_Of_Month'] = week
        
        future_rows.append(mean_features)
        
    forecast_df = pd.DataFrame(future_rows)
    forecast_df['Predicted_Price'] = model.predict(forecast_df[features])
    return forecast_df

# Dry run
test_market = "Ahmednagar APMC"
f_res = generate_forecast(test_market)
print(f"=== Future Price Forecast for {test_market} ===")
for idx, row in f_res.iterrows():
    print(f"Week {int(row['time_index'])} (Month {int(row['Month_Num'])}, Week {int(row['Week_Of_Month'])}): Rs {row['Predicted_Price']:.2f}/quintal")

In [ ]:
# Cell 3: Spatio-Temporal Arbitrage (Best Market Recommendation)
def get_market_recommendations(market_name, target_week=289):
    """Compares predicted prices for all markets in the same district and ranks them"""
    market_df = df[df['Market'] == market_name]
    if market_df.empty:
        return pd.DataFrame()
    
    district = market_df['District'].iloc[0]
    district_markets = df[df['District'] == district]['Market'].unique()
    
    recommendations = []
    for dm in district_markets:
        f_df = generate_forecast(dm)
        row = f_df[f_df['time_index'] == target_week]
        if not row.empty:
            recommendations.append({
                'Market': dm,
                'Predicted_Price': row['Predicted_Price'].iloc[0]
            })
            
    rec_df = pd.DataFrame(recommendations).sort_values('Predicted_Price', ascending=False)
    # Calculate price difference premium compared to the selected market
    base_price = rec_df[rec_df['Market'] == market_name]['Predicted_Price'].iloc[0]
    rec_df['Price_Premium'] = rec_df['Predicted_Price'] - base_price
    return rec_df, district

# Dry run
rec_res, dist_name = get_market_recommendations("Ahmednagar APMC")
print(f"=== Best Markets in {dist_name} District (Week 289) ===")
print(rec_res.to_string(index=False))

In [ ]:
# Cell 4: Price Crash Alerts Engine
def get_price_crash_alerts(target_week=289, threshold_pct=10.0):
    """Scans all markets for predicted drop of > threshold_pct compared to week 288"""
    alerts = []
    for m in df['Market'].unique():
        hist_last = df[(df['Market'] == m) & (df['time_index'] == 288)]['Price_Rs_Per_Quintal'].iloc[0]
        f_df = generate_forecast(m)
        row = f_df[f_df['time_index'] == target_week]
        if not row.empty:
            pred_val = row['Predicted_Price'].iloc[0]
            drop_pct = ((hist_last - pred_val) / hist_last) * 100
            if drop_pct > threshold_pct:
                alerts.append({
                    'Market': m,
                    'District': df[df['Market'] == m]['District'].iloc[0],
                    'Last_Price_W288': hist_last,
                    'Predicted_Price_W289': pred_val,
                    'Drop_Pct': drop_pct
                })
                
    if not alerts:
        return pd.DataFrame(columns=['Market', 'District', 'Last_Price_W288', 'Predicted_Price_W289', 'Drop_Pct'])
    return pd.DataFrame(alerts).sort_values('Drop_Pct', ascending=False)

# Dry run
alerts_res = get_price_crash_alerts(threshold_pct=10.0)
print(f"=== High Risk Price Crash Markets (Week 289) ===")
if alerts_res.empty:
    print("No alerts triggered.")
else:
    print(alerts_res.head(5).to_string(index=False))

In [ ]:
# Cell 5: Climate Simulator Engine
def generate_climate_forecast(market_name, rain_change_pct, temp_change_val):
    """Generates 4 weeks of future forecasts under simulated climate variations"""
    forecast_df = generate_forecast(market_name)
    if forecast_df.empty:
        return pd.DataFrame()
        
    sim_df = forecast_df.copy()
    
    # Scale rainfall variables
    rain_cols = ['Rainfall', 'rain_lag_1', 'rain_lag_2', 'rain_lag_4', 'rain_lag_8', 
                 'rain_cum_4w', 'rain_cum_8w', 'rain_cum_12w', 'monsoon_total', 'monsoon_prev_year']
    for col in rain_cols:
        sim_df[col] = sim_df[col] * (1.0 + rain_change_pct / 100.0)
        
    # Shift temperature variables
    temp_cols = ['temp_mean', 'temp_min', 'temp_max', 'temp_mean_lag_1', 'temp_mean_lag_2', 
                 'temp_mean_lag_3', 'temp_mean_lag_4', 'temp_mean_roll_2', 'temp_mean_roll_4']
    for col in temp_cols:
        sim_df[col] = sim_df[col] + temp_change_val
        
    sim_df['Predicted_Price'] = model.predict(sim_df[features])
    return sim_df

# Dry run: Drought (-50% Rain) and Hot (+3.0 C Temp)
rain_sim, temp_sim = -50.0, 3.0
sim_res = generate_climate_forecast("Ahmednagar APMC", rain_sim, temp_sim)
print(f"=== Climate Simulation: Ahmednagar APMC ({rain_sim}% rain, {temp_sim:+.1f}C temp) ===")
for t, bp, sp in zip(f_res['time_index'], f_res['Predicted_Price'], sim_res['Predicted_Price']):
    diff_pct = ((sp - bp) / bp) * 100
    print(f"Week {int(t)}: Baseline = Rs {bp:.2f}, Simulated = Rs {sp:.2f} ({diff_pct:+.2f}%)")

In [ ]:
# Cell 6: Chatbot NLP Parsing Engine
def chatbot_query(query):
    q = query.lower()
    
    # 1. Price Crash Alerts
    if any(k in q for k in ['crash', 'alert', 'drop', 'warning', 'fall']):
        alerts = get_price_crash_alerts(target_week=289, threshold_pct=10.0)
        if alerts.empty:
            return "✅ No major price crash alerts next week."
        else:
            ans = "⚠️ **Price Crash Alerts for Next Week:**\n"
            for idx, row in alerts.head(3).iterrows():
                ans += f"- **{row['Market']}**: drop of **-{row['Drop_Pct']:.1f}%** (Rs {row['Predicted_Price_W289']:.2f} vs Rs {row['Last_Price_W288']:.2f})\n"
            return ans
            
    # 2. Recommendations
    if any(k in q for k in ['recommend', 'best market', 'where to sell', 'highest price', 'sell']):
        matched_m = None
        for m in df['Market'].unique():
            core = m.replace(" APMC", "").lower()
            if core in q:
                matched_m = m
                break
        if matched_m:
            rec, dist = get_market_recommendations(matched_m)
            best_m = rec.iloc[0]['Market']
            best_p = rec.iloc[0]['Predicted_Price']
            prem = rec.iloc[0]['Price_Premium']
            return f"🏆 In **{dist}** district, the highest predicted price is at **{best_m}** (Rs {best_p:.2f}/q). Selling there gives you a premium of **+Rs {prem:.2f}/q** compared to {matched_m}."
        else:
            return "Please include the name of your local market so I can search your district's recommendations (e.g. 'best market to sell near Ahmednagar')."
            
    # Default fallback
    return "👋 Hello! I can help you find price forecasts, crash warnings, and best selling markets. Try asking: 'Are there price crash alerts next week?' or 'recommend where to sell near Ahmednagar'."

# Dry runs
print("=== Chatbot Dry Run 1 (Alerts) ===")
print(chatbot_query("show me price crash alerts"))
print("\n=== Chatbot Dry Run 2 (Recommendations) ===")
print(chatbot_query("where should I sell near Ahmednagar APMC?"))